# Batch satellite image fetch

Collects building footprints around an address, filters them by size, saves them as
GeoJSON, fetches one satellite image per house, and merges the result into
`manifest.csv` along with whether each roof already has solar panels.

Images are keyed by the polygon id `b{x0}_{y0}_{w}x{h}` (world pixels at `ID_ZOOM`),
so the id encodes the crop box and every image comes out exactly `w x h` pixels.

Imagery goes through `SatelliteTileCache`, so overlapping houses share tiles: in a
typical suburban block ~170 houses need only ~40 tile downloads, and re-runs cost
nothing.

**Run cells top to bottom** (Kernel -> Restart Kernel and Run All Cells).
Steps 1-4 need no API key. Step 5 needs `GOOGLE_MAPS_API_KEY`; step 6 needs it only
for houses whose solar lookup is not already cached, and prints how many that is
before spending anything.

In [ ]:
import json
import os
import re
import statistics
import sys
from pathlib import Path

sys.path.append("../scripts")

from get_building_footprint import (
    geocode_address,
    query_overpass_buildings,
    elements_to_polygons,
    polygon_area_m2,
    polygon_area_sqft,
)
from satellite_cache import (
    SatelliteTileCache,
    polygon_id,
    polygon_id_bbox,
    parse_polygon_id,
    GOOGLE_ATTRIBUTION,
)

HOUSE_DIR = Path("../data/house")
IMAGE_DIR = HOUSE_DIR / "images"
HOUSE_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Get building polygons for an address and radius

In [ ]:
address = "1095 HAPPY VALLEY AVE, SAN JOSE, CA"
radius_m = 200

lat, lon = geocode_address(address)
elements = query_overpass_buildings(lat, lon, radius_m)
polygons = elements_to_polygons(elements)

print(f"{address}")
print(f"  centre   : {lat}, {lon}")
print(f"  radius   : {radius_m} m")
print(f"  buildings: {len(polygons)}")

## 2. Filter by footprint size and address

Bulk-imported OSM data mixes houses with sheds, garages and apartment blocks, so
trim to a plausible size range before fetching imagery.

`REQUIRE_ADDRESS` additionally drops footprints with no `addr:housenumber` +
`addr:street`. Unaddressed buildings are usually outbuildings or partially-mapped
records, and without an address they cannot be joined to any external data — so they
are dead weight in a labelled dataset.

Note the size is **footprint** area — the ground outline. It is not a listing's
"square footage", which counts every storey and excludes the garage. A two-storey
2400 sqft house has a ~1200 sqft footprint.

In [ ]:
MIN_SQFT = 800
MAX_SQFT = 6000
REQUIRE_ADDRESS = True  # drop footprints OSM has no street address for

houses = []
dropped_size = 0
dropped_address = 0

for element, poly in polygons:
    tags = element.get("tags", {})

    sqft = polygon_area_sqft(poly)
    if not (MIN_SQFT <= sqft <= MAX_SQFT):
        dropped_size += 1
        continue

    # both parts are required -- a lone city or postcode does not identify a house
    has_address = bool(tags.get("addr:housenumber") and tags.get("addr:street"))
    if REQUIRE_ADDRESS and not has_address:
        dropped_address += 1
        continue

    houses.append({
        "id": polygon_id(poly),
        "osm_id": element.get("id"),
        "osm_type": element.get("type", "way"),
        "area_sqft": round(sqft, 1),
        "area_m2": round(polygon_area_m2(poly), 1),
        "tags": tags,
        "polygon": poly,
    })

print(f"kept {len(houses)} of {len(polygons)} buildings")
print(f"  dropped, outside {MIN_SQFT}-{MAX_SQFT} sqft : {dropped_size}")
print(f"  dropped, no street address           : {dropped_address}")

# ids must be unique -- two footprints sharing a quantised bounding box would
# otherwise overwrite each other's image
ids = [h["id"] for h in houses]
assert len(set(ids)) == len(ids), f"duplicate ids: {len(ids) - len(set(ids))}"

## 3. Statistics

In [ ]:
areas = [h["area_sqft"] for h in houses]

if not areas:
    print("no houses passed the filter -- widen MIN_SQFT/MAX_SQFT")
else:
    print(f"houses        : {len(areas)}")
    print(f"min footprint : {min(areas):>8,.0f} sqft")
    print(f"max footprint : {max(areas):>8,.0f} sqft")
    print(f"mean          : {statistics.mean(areas):>8,.0f} sqft")
    print(f"median        : {statistics.median(areas):>8,.0f} sqft")
    if len(areas) > 1:
        print(f"stdev         : {statistics.stdev(areas):>8,.0f} sqft")

    print()
    lo, hi = min(areas), max(areas)
    nbins, width = 10, max((hi - lo) / 10, 1e-9)
    for b in range(nbins):
        edge = lo + b * width
        n = sum(1 for a in areas if edge <= a < edge + width
                or (b == nbins - 1 and a == hi))
        print(f"  {edge:>6,.0f}-{edge + width:>6,.0f} sqft | {'#' * n}{'' if n else ''} {n}")

## 4. Save the polygons to `data/house/`

Written as GeoJSON so it opens in any GIS tool, with the polygon id, OSM id and area
carried in each feature's properties.

In [ ]:
def slugify(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

json_name = f"{slugify(address)}_r{radius_m}.json"
json_path = HOUSE_DIR / json_name

feature_collection = {
    "type": "FeatureCollection",
    "properties": {
        "address": address,
        "center": [lat, lon],
        "radius_m": radius_m,
        "min_sqft": MIN_SQFT,
        "max_sqft": MAX_SQFT,
        "require_address": REQUIRE_ADDRESS,
    },
    "features": [
        {
            "type": "Feature",
            "properties": {k: v for k, v in h.items() if k != "polygon"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[[x, y] for x, y in h["polygon"].exterior.coords]],
            },
        }
        for h in houses
    ],
}

with open(json_path, "w") as f:
    json.dump(feature_collection, f, indent=2)

print(f"wrote {len(houses)} houses to {json_path}")

## 5. Fetch a satellite image per house

Reads the file written above, crops each house to its **bounding box**, and saves
`{id}_raw.png`. The separator is an underscore rather than a dot so the filename has a
single suffix -- `Path.stem` then yields `{id}_raw` cleanly, where `{id}.raw.png` would
leave a stray `.raw` behind.

Because the id encodes the box in world pixels, each image is exactly `w x h` pixels --
verified below.

The tile count is previewed first: only uncached tiles cost an API call, and houses on
the same block share them.

In [ ]:
with open(json_path) as f:
    saved = json.load(f)

features = saved["features"]
cache = SatelliteTileCache(cache_dir="../data/tiles")

# Preview cost: how many distinct tiles does the whole batch touch?
needed = set()
for feat in features:
    bbox = polygon_id_bbox(feat["properties"]["id"])
    i0, j0, i1, j1 = cache.tile_indices_for_bbox(bbox)
    needed.update((i, j) for i in range(i0, i1 + 1) for j in range(j0, j1 + 1))

uncached = [t for t in needed if not cache.tile_path(*t).exists()]
print(f"{len(features)} houses -> {len(needed)} distinct tiles, "
      f"{len(uncached)} not yet cached ({len(uncached)} API calls)")

In [ ]:
failures = []
written = 0
skipped = 0

for feat in features:
    house_id = feat["properties"]["id"]
    out_path = IMAGE_DIR / f"{house_id}_raw.png"
    if out_path.exists():
        skipped += 1
        continue
    try:
        image, _ = cache.get_region(polygon_id_bbox(house_id))
        x0, y0, x1, y1 = parse_polygon_id(house_id)
        if image.size != (x1 - x0, y1 - y0):
            raise ValueError(f"got {image.size}, id encodes {(x1 - x0, y1 - y0)}")
        image.save(out_path)
        written += 1
    except Exception as exc:
        failures.append((house_id, repr(exc)))

print(f"written {written}, already present {skipped}, failed {len(failures)}")
print(f"tiles: {cache.downloads} downloaded, {cache.hits} from cache")
print(f"images in {IMAGE_DIR}  -- {GOOGLE_ATTRIBUTION}")

for house_id, err in failures[:10]:
    print("  FAILED", house_id, err)

### Spot-check a few of the saved crops

In [ ]:
import matplotlib.pyplot as plt

saved_images = sorted(IMAGE_DIR.glob("*_raw.png"))
print(f"{len(saved_images)} images on disk")

sample = saved_images[:8]
if sample:
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)
    flat = axes.ravel()
    for ax, path in zip(flat, sample):
        img = plt.imread(path)
        ax.imshow(img)
        house_id = path.stem.removesuffix("_raw")
        ax.set_title(f"{house_id.split('_')[-1]}\n{img.shape[1]}x{img.shape[0]} px",
                     fontsize=7)
        ax.axis("off")
    for ax in flat[len(sample):]:
        ax.axis("off")
    fig.suptitle(f"{address} -- {GOOGLE_ATTRIBUTION}", fontsize=9)
    plt.tight_layout()
    plt.show()

## 6. Update `manifest.csv` with solar info

Image filenames are polygon ids, which carry no hint of which house they are, so the
manifest is the lookup table beside them: find a house by address, see which ones
still lack imagery, or load ids, areas and panel labels as model targets.

This **merges** rather than overwrites. Earlier runs — other addresses, other radii —
keep their rows, and only houses not already in the file are appended. That matters
because the manifest accumulates columns no single run can rebuild: re-deriving it
from `features` alone would silently drop every solar column.

Existing rows are left as they are, with two exceptions:

- `has_image` is re-checked against disk, since it describes the filesystem and goes
  stale the moment step 5 fetches anything.
- Blank solar columns are filled in. Values already present are never overwritten, so
  a manual correction survives a re-run.

Solar data comes from `data/house/solar/`, the cache written by
`scripts/get_solar_insights.py`. Anything already cached is free; only houses never
looked up cost an API call, and the count is printed before it spends. Set
`FETCH_SOLAR = False` to fill from cache only.

Rows with no image sort last, so the tail of the file is still the to-do list.

In [ ]:
from get_building_footprint import format_address
from get_solar_insights import manifest_row, merge_manifest

FETCH_SOLAR = True  # False fills solar columns from cache only, spending nothing

manifest_path = HOUSE_DIR / "manifest.csv"
SOLAR_DIR = HOUSE_DIR / "solar"

# Everything geometric comes from the polygon id, so only the columns the id cannot
# encode are passed through.
candidates = [
    manifest_row(
        p["id"],
        address=format_address(p.get("tags", {})),
        area_sqft=p.get("area_sqft", ""),
        area_m2=p.get("area_m2", ""),
        osm_type=p.get("osm_type", ""),
        osm_id=p.get("osm_id", ""),
    )
    for p in (feat["properties"] for feat in features)
]

rows, added, filled = merge_manifest(
    manifest_path,
    candidates,
    solar_dir=SOLAR_DIR,
    api_key=os.environ.get("GOOGLE_MAPS_API_KEY"),
    image_dir=IMAGE_DIR,
    fetch=FETCH_SOLAR,
    log=print,
)

with_image = sum(int(r["has_image"]) for r in rows)
with_panels = sum(1 for r in rows if r.get("has_panels") == "1")
known_panels = sum(1 for r in rows if r.get("has_panels") in ("0", "1"))
print(f"\nwrote {len(rows)} rows to {manifest_path}")
print(f"  with image  : {with_image}")
print(f"  missing     : {len(rows) - with_image}")
print(f"  addressed   : {sum(1 for r in rows if r['address'])}")
print(f"  with panels : {with_panels} of {known_panels} with a solar verdict")